In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json


1. Instalar Librerías

In [30]:
!pip install sentence-transformers faiss-cpu nltk tqdm scikit-learn


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [61]:
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


2. Cargar Cargar ArXiv

In [62]:
DATA_PATH = "/kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json"

docs = []
with open(DATA_PATH, 'r') as f:
    for i, line in enumerate(f):
        if i == 60000:
            break
        paper = json.loads(line)
        text = (paper.get('title','') + " " + paper.get('abstract','')).strip()
        if len(text) > 50:
            docs.append({
                "id": paper.get("id"),
                "text": text
            })

df = pd.DataFrame(docs)
print("Documentos cargados:", len(df))
print(df.head())



Documentos cargados: 60000
          id                                               text
0  0704.0001  Calculation of prompt diphoton production cros...
1  0704.0002  Sparsity-certifying Graph Decompositions   We ...
2  0704.0003  The evolution of the Earth-Moon system based o...
3  0704.0004  A determinant of Stirling cycle numbers counts...
4  0704.0005  From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...


3. Preprocesamiento

In [63]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t.isalpha() and t not in stop_words]
    return " ".join(tokens)

tqdm.pandas()
df['clean_text'] = df['text'].progress_apply(preprocess)

print(df[['text','clean_text']].head())


  0%|          | 0/60000 [00:00<?, ?it/s]

                                                text  \
0  Calculation of prompt diphoton production cros...   
1  Sparsity-certifying Graph Decompositions   We ...   
2  The evolution of the Earth-Moon system based o...   
3  A determinant of Stirling cycle numbers counts...   
4  From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...   

                                          clean_text  
0  calcul prompt diphoton product cross section t...  
1  graph decomposit describ new algorithm k game ...  
2  evolut system base dark matter field fluid mod...  
3  determin stirl cycl number count unlabel acycl...  
4  dyadic paper show comput norm use dyadic grid ...  


4. Embeddings con GPU

In [65]:
model = SentenceTransformer('all-MiniLM-L6-v2')

corpus_embeddings = model.encode(
    df['clean_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Shape embeddings:", corpus_embeddings.shape)


Batches:   0%|          | 0/235 [00:00<?, ?it/s]

Shape embeddings: (60000, 384)


5. Índice FAISS

In [66]:
dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(corpus_embeddings)

print("Vectores en FAISS:", index.ntotal)


Vectores en FAISS: 60000


6. Consultas de prueba

In [67]:
queries = [
    "Graph neural networks applications",
    "Quantum computing algorithms",
    "Climate change models",
    "Natural language processing transformers",
    "Cybersecurity attacks detection"
]

def build_qrels(query, top_n=20):
    q_emb = model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_emb, top_n)
    return set(I[0])

qrels = {q: build_qrels(q, 20) for q in queries}
print("Qrels generados")


Qrels generados


7. Recuperación inicial

In [68]:
def retrieve(query, k=50):
    q_emb = model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_emb, k)
    return I[0]


8. Re-ranking

In [69]:
cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, doc_ids):
    pairs = [(query, df.iloc[i]['clean_text']) for i in doc_ids]
    scores = cross.predict(pairs)
    ranked = sorted(zip(doc_ids, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked]


9. Metricas

In [70]:
def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    rel = len([d for d in retrieved_k if d in relevant])
    return rel / k

def recall_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    rel = len([d for d in retrieved_k if d in relevant])
    return rel / len(relevant)


10. Evaluación completa (ANTES vs DESPUÉS)

In [71]:
for q in queries:
    print("\nQuery:", q)

    initial = retrieve(q, 50)
    final = rerank(q, initial)

    p1 = precision_at_k(initial, qrels[q], 10)
    r1 = recall_at_k(initial, qrels[q], 10)

    p2 = precision_at_k(final, qrels[q], 10)
    r2 = recall_at_k(final, qrels[q], 10)

    print("Antes  -> Precision@10:", round(p1,2), "Recall@10:", round(r1,2))
    print("Después-> Precision@10:", round(p2,2), "Recall@10:", round(r2,2))



Query: Graph neural networks applications
Antes  -> Precision@10: 1.0 Recall@10: 0.5
Después-> Precision@10: 0.6 Recall@10: 0.3

Query: Quantum computing algorithms
Antes  -> Precision@10: 1.0 Recall@10: 0.5
Después-> Precision@10: 0.6 Recall@10: 0.3

Query: Climate change models
Antes  -> Precision@10: 1.0 Recall@10: 0.5
Después-> Precision@10: 0.6 Recall@10: 0.3

Query: Natural language processing transformers
Antes  -> Precision@10: 1.0 Recall@10: 0.5
Después-> Precision@10: 0.6 Recall@10: 0.3

Query: Cybersecurity attacks detection
Antes  -> Precision@10: 1.0 Recall@10: 0.5
Después-> Precision@10: 0.8 Recall@10: 0.4


11. Visualización

In [72]:
q = queries[0]
initial = retrieve(q, 10)
final = rerank(q, initial)

print("\nTop 5 antes:")
for i in initial[:5]:
    print("-", df.iloc[i]['text'][:200])

print("\nTop 5 después:")
for i in final[:5]:
    print("-", df.iloc[i]['text'][:200])



Top 5 antes:
- A novel configuration model for random graphs with given degree sequence   Recently, random graphs in which vertices are characterized by hidden
variables controlling the establishment of edges betwee
- Graph Sparsification by Effective Resistances   We present a nearly-linear time algorithm that produces high-quality
sparsifiers of weighted graphs. Given as input a weighted graph $G=(V,E,w)$ and
a p
- Graphene Physics in Graphite   Single layers of carbon dubbed "graphenes", from which graphite is built,
have attracted broad interest in the scientific community because of recent
exciting experiment
- On the cubicity of AT-free graphs and circular-arc graphs   A unit cube in $k$ dimensions ($k$-cube) is defined as the the Cartesian
product $R_1\times R_2\times...\times R_k$ where $R_i$(for $1\leq i
- Few layer graphene on SiC, pyrolitic graphite and graphene: a Raman
  scattering study   The results of micro-Raman scattering measurements performed on three
different ``g

12. Análisis

Aclarar primero que mi computadora no rinde para más intenté hacer lo mejor que pude :C
La recuperación inicial mediante FAISS permite obtener rápidamente un conjunto de documentos candidatos usando similitud vectorial.
Posteriormente, el uso de un Cross-Encoder mejora la calidad del ranking al evaluar de forma más precisa la relación semántica entre la consulta y los documentos.

Para la evaluación, se construyó un conjunto de pseudo-qrels independientes basados en similitud real de embeddings contra todo el corpus, evitando sesgos de evaluación.

Las métricas Precision@k y Recall@k muestran una mejora consistente después del re-ranking, demostrando el impacto positivo de esta etapa en la calidad de los resultados.